In [ ]:
import timm
import torch
import numpy
import pandas
import requests
import torchvision
import torch.nn as nn
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from datasets import load_dataset
from torchvision import transforms
from typing import Tuple, Dict, List
import torchvision.datasets as datasets
from timeit import default_timer as Timer
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, DataLoader, Dataset

In [ ]:
device= "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
model = model.to(device)

In [ ]:
dataset = load_dataset("mikehemberger/plantnet300K")

In [ ]:
dataset

In [ ]:
dataset['test']

In [ ]:
print(dataset["train"].features["label"])

In [ ]:
label_names= dataset["train"].features["label"].names
label_names

In [ ]:
# they host metadata files separately here
url = "https://seafile.plantnet.org/d/bed81bc15e8944969cf6/files/?p=%2Fplantnet300K_species_id_2_name.json&dl=1"
r = requests.get(url)
print(r.status_code)
print(r.text[:200])   # print first 200 chars to see what's coming back

In [ ]:
mapping = r.json()
print(list(mapping.items()))

In [ ]:
len(mapping)

In [ ]:
import json

with open("plantnet_mapping.json", "w") as f:
    json.dump(mapping, f)

print(f"Saved. Total species: {len(mapping)}")

In [ ]:
label_name= dataset["train"].features["label"].names

def idx_to_names(idx):
  species_id= label_names[idx]
  return mapping.get(species_id, species_id)

print(idx_to_names(0))
print(idx_to_names(1))
print(idx_to_names(2))

In [ ]:
plnt_name=[mapping[species_id] for species_id in label_name]
len(plnt_name)

In [ ]:
dataset["train"][2]

In [ ]:
plt.imshow(dataset["train"][9561]["image"])
plt.title(plnt_name[dataset["train"][9561]["label"]])
plt.show()

# Transform

In [ ]:
train_transform=transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_val_transform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])

])

class PlantNet(Dataset):
  def __init__(self, hf_dataset, transform=None):
    self.data=hf_dataset
    self.transform=transform

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    img= self.data[idx]["image"].convert("RGB")
    lbl= self.data[idx]["label"]
    if self.transform:
      img=self.transform(img)
    return img, lbl

# Data Sets

In [ ]:
train_set = PlantNet(dataset["train"], transform=train_transform)
val_set= PlantNet(dataset["validation"], transform=test_val_transform)
test_set= PlantNet(dataset["test"], transform=test_val_transform)

# Data Loader

In [ ]:
train_loader= DataLoader(train_set,
                        batch_size=32,
                        shuffle=True,
                        num_workers=2)
val_loader= DataLoader(val_set,
                        batch_size=32,
                        shuffle=False,
                        num_workers=2)
test_loader= DataLoader(test_set,
                        batch_size=32,
                        shuffle=False,
                        num_workers=2)

print(f"Train: {len(train_set)} | Val: {len(val_set)} |  Test: {len(test_set)}")

In [ ]:
test_set[9]

In [ ]:
print(train_transform)

In [ ]:
img, label = test_set[9]
print(img.shape)   # should be torch.Size([3, 224, 224])
print(label)       # should be an integer
print(plnt_name[label])  # should print a plant name

In [ ]:
fig= plt.figure(figsize=(9,9))
rw, cl= 4,4
for i in range(1, rw*cl+1):
  ridx=torch.randint(0, len(train_loader), size=[1]).item()
  img, lbl= train_set[ridx]
  fig.add_subplot(rw, cl, i)
  plt.imshow(img.permute(1,2,0))
  plt.title(plnt_name[lbl])
  plt.axis(False)

# Model

In [ ]:
from timm.models.efficientnet import efficientnet_b0
model= timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=1081
)

model=model.to(device)

In [ ]:
opti=torch.optim.Adam(
    filter(lambda p:p.requires_grad, model.parameters()),
    lr=1e-3)

In [ ]:
def Time(start: float, end: float, device: torch.device=None):
  total_time=end-start
  print(f"Total Time: {total_time:.3f}secs")
  return total_time

In [ ]:
def Model_Eval_1(model,
               train_dataloader,
               val_dataloader,
               test_dataloader,
               loss_fnc,
               optimizer,
               n,
               seed,
               device=device):

    start = Timer()
    results = {"train_loss": [], "train_acc": [],
               "val_loss":   [], "val_acc":   [],
               "test_loss":  [], "test_acc":  []}

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n)
    best_acc = 0

    def train_test(model, train_dataloader, val_dataloader, loss_fnc, optimizer, device):

        # Train
        model.train()
        train_loss, train_acc = 0, 0
        for batch, (x, y) in enumerate(train_dataloader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            y_pred = model(x)
            loss1 = loss_fnc(y_pred, y)
            loss1.backward()
            optimizer.step()
            train_loss += loss1.item()
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            train_acc += (y_pred_class == y).sum().item() / len(y_pred)
        train_loss = train_loss / len(train_dataloader)
        train_acc  = train_acc  / len(train_dataloader)

        # Val
        model.eval()
        val_loss, val_acc = 0, 0
        with torch.inference_mode():
            for batch, (x, y) in enumerate(val_dataloader):
                x, y = x.to(device), y.to(device)
                val_pred = model(x)
                loss2 = loss_fnc(val_pred, y)
                val_loss += loss2.item()
                val_acc += (val_pred.argmax(dim=1) == y).sum().item() / len(y)
        val_loss = val_loss / len(val_dataloader)
        val_acc  = val_acc  / len(val_dataloader)

        return train_loss, train_acc, val_loss, val_acc

    torch.manual_seed(seed)

    for i in tqdm(range(n)):
        start = Timer()
        print(f"EPOCH: {i+1}\n---------------")
        train_loss, train_acc, val_loss, val_acc = train_test(
            model, train_dataloader, val_dataloader, loss_fnc, optimizer, device
        )
        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"Train Acc:  {train_acc:.4f}  | Val Acc:  {val_acc:.4f}")

        scheduler.step()

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")
            print(f"  ✓ Best model saved (val acc: {best_acc:.4f})")

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)
        end = Timer()
        Time(start, end)

    # final test evaluation after all epochs
    print("\n── Final Test Evaluation ──")
    model.load_state_dict(torch.load("best_model.pth"))
    model.eval()
    test_loss, test_acc = 0, 0
    with torch.inference_mode():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            test_loss += loss_fnc(pred, y).item()
            test_acc  += (pred.argmax(dim=1) == y).sum().item() / len(y)
    test_loss = test_loss / len(test_dataloader)
    test_acc  = test_acc  / len(test_dataloader)
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

    end = Timer()
    Time(start, end)
    print(f"\nBest Val Accuracy: {best_acc:.4f}")

    return results

In [ ]:
try:
  import torchinfo
except:
  !pip install torchinfo
  import torchinfo

from torchinfo import summary
summary(model, input_size=[1,3,244,244])

In [ ]:
model_res=Model_Eval_1(model, train_loader, val_loader, test_loader, loss_fn, opti, 10, 42)

In [ ]:
def plot_loss_curve(result:  Dict[str, List[float]]):

  def to_float(data_list):
    clean_list = []
    for item in data_list:
      if torch.is_tensor(item):
        # This handles the exact error: detach from graph, move to cpu, make float
        clean_list.append(item.detach().cpu().item())
      else:
        clean_list.append(float(item))
    return clean_list

  # Convert all 4 lists forcefully
  loss = to_float(result['train_loss'])
  test_loss = to_float(result['test_loss'])
  acc = to_float(result['train_acc'])
  test_acc = to_float(result['test_acc'])

  acc=result['train_acc']
  test_acc=result['test_acc']

  epoch=range(len(result['train_loss']))
  plt.figure(figsize=(15,7))

  plt.subplot(1,2,1)
  plt.plot(epoch, loss, label='train_loss')
  plt.plot(epoch, test_loss, label='test_loss')
  plt.title('Loss')
  plt.xlabel('Epochs')
  plt.legend()

  plt.subplot(1,2,2)
  plt.plot(epoch, acc, label='train_acc')
  plt.plot(epoch, test_acc,label='test_acc')
  plt.title('Acc')
  plt.xlabel('Epochs')
  plt.legend();

In [ ]:
plot_loss_curve(model_res)